# Bandarlog · generador de ilustraciones en Colab gratuito
Ejecuta las celdas en orden. El cuaderno utiliza la rama `main` del repositorio por defecto. Funciona con CPU; no requiere contratar Colab Pro ni una API de pago. Google limita la duración y disponibilidad de sus sesiones gratuitas. Si la sesión termina, el generador deja de estar disponible y el enlace temporal cambia al reiniciarlo.

La aplicación web y Neon siguen en Railway. Este cuaderno solo genera las ilustraciones de comisiones. No se envían nombres de personas, tareas ni fotos de perfil al modelo. No guardes claves en celdas, salidas ni GitHub.


In [ ]:
from pathlib import Path
import subprocess, sys, os
REF = 'main' #@param {type:"string"}
repo = Path('/content/bandarlog')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, 'https://github.com/nunudiewillump/bandarlog.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', REF], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
assert (repo / 'ai-worker/model.py').exists(), 'La rama elegida todavía no incluye el generador de IA.'
venv = Path('/content/bandarlog-ai-venv')
bootstrap = Path('/content/bandarlog-bootstrap')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', str(bootstrap), 'uv==0.12.23'], check=True)
uv = str(bootstrap / 'bin/uv')
python = str(venv / 'bin/python')
usable = False
if Path(python).is_file():
    probe = subprocess.run([python, '-c', 'import sys, pip; assert sys.version_info[:2] == (3, 12)'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    usable = probe.returncode == 0
if not usable:
    subprocess.run([uv, 'venv', '--python', '3.12', '--managed-python', '--seed', '--clear', '--system-certs', str(venv)], check=True, env={**os.environ, 'UV_CACHE_DIR': '/content/bandarlog-uv-cache', 'UV_PYTHON_INSTALL_DIR': '/content/bandarlog-python'})
subprocess.run([python, '-m', 'pip', 'install', '-r', str(repo / 'ai-worker/requirements.txt')], check=True)


## Clave privada y arranque
Elige una clave aleatoria larga en tu gestor de contraseñas. Introdúcela abajo y guárdala también en Railway como `ARTWORK_WORKER_TOKEN`. El campo no muestra ni guarda la clave en la salida del cuaderno.


In [ ]:
from getpass import getpass
from pathlib import Path
import os, subprocess, time, urllib.request, json
repo = Path('/content/bandarlog')
python = str(Path('/content/bandarlog-ai-venv/bin/python'))
if not (repo / 'ai-worker/model.py').is_file() or not Path(python).is_file():
    raise RuntimeError('Ejecuta primero la celda de instalación y espera a que termine sin errores. Después vuelve a ejecutar esta celda.')
previous_worker = globals().get('worker')
if previous_worker is not None and previous_worker.poll() is None:
    previous_worker.terminate()
    previous_worker.wait(timeout=15)
worker_token = getpass('Clave privada del generador (mínimo 24 caracteres): ')
assert len(worker_token) >= 24, 'Utiliza una clave aleatoria de al menos 24 caracteres.'
worker_env = {**os.environ, 'ARTWORK_WORKER_TOKEN': worker_token, 'HF_HOME': '/content/bandarlog-model-cache', 'U2NET_HOME': '/content/bandarlog-model-cache/u2net', 'HF_HUB_DISABLE_XET': '1', 'AI_THREADS': '2'}
worker_log = open('/content/bandarlog-worker.log', 'w')
worker = subprocess.Popen([python, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '7860', '--no-access-log'], cwd=str(repo / 'ai-worker'), env=worker_env, stdout=worker_log, stderr=subprocess.STDOUT)
for attempt in range(60):
    try:
        health = json.load(urllib.request.urlopen('http://127.0.0.1:7860/health', timeout=3))
        if health.get('ok') and health.get('configured'): break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('El generador no ha arrancado. Revisa el registro local sin compartir claves.')
print('Generador preparado; la clave no se muestra.')


## Enlace HTTPS temporal para Railway
Se abre un túnel de prueba gratuito de Cloudflare. Puedes probar la generación sin modificar Railway. El generador sigue protegido por la clave privada. Solo se muestra la dirección pública; la descarga del ejecutable comprueba el SHA-256 publicado por GitHub.


In [ ]:
import hashlib, re, os, subprocess, time, urllib.request, json
from pathlib import Path

def install_cloudflared(path, data):
    # Replace the directory entry, never truncate an executable still in use.
    import os, tempfile
    from pathlib import Path
    staged = None
    try:
        with tempfile.NamedTemporaryFile(dir=path.parent, prefix=path.name + '.', delete=False) as output:
            staged = Path(output.name)
            output.write(data)
        staged.chmod(0o700)
        os.replace(staged, path)
    finally:
        if staged is not None:
            staged.unlink(missing_ok=True)

release = json.load(urllib.request.urlopen('https://api.github.com/repos/cloudflare/cloudflared/releases/latest'))
asset = next(a for a in release['assets'] if a['name'] == 'cloudflared-linux-amd64')
digest = asset.get('digest', '')
assert digest.startswith('sha256:'), 'Falta el SHA-256 oficial del ejecutable; no se ejecutará la descarga.'
data = urllib.request.urlopen(asset['browser_download_url']).read()
assert hashlib.sha256(data).hexdigest() == digest.split(':', 1)[1], 'La verificación SHA-256 ha fallado.'
cloudflared = Path('/content/bandarlog-cloudflared')
install_cloudflared(cloudflared, data)
previous_tunnel = globals().get('tunnel')
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_tunnel.kill()
        previous_tunnel.wait(timeout=5)
previous_log = globals().get('tunnel_log')
if previous_log is not None:
    previous_log.close()
globals().pop('worker_url', None)
tunnel_log_path = Path('/content/bandarlog-tunnel.log')
tunnel_log = tunnel_log_path.open('w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
for attempt in range(60):
    if tunnel.poll() is not None:
        raise RuntimeError('El túnel se ha detenido. Revisa el registro del túnel antes de reintentarlo.')
    matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log_path.read_text())
    if matches:
        worker_url = matches[-1]
        break
    time.sleep(1)
else:
    raise RuntimeError('No se ha obtenido un enlace temporal. El proveedor puede no estar disponible.')
print('URL temporal: actualiza ARTWORK_WORKER_URL en Railway si ha cambiado:')
print(worker_url)
print('ARTWORK_WORKER_TOKEN debe contener la clave que has introducido, sin compartirla aquí.')


## Prueba opcional de generación
La primera ejecución descarga los modelos y puede tardar varios minutos. Los siguientes resultados se generan en la CPU gratuita de esta sesión. La prueba usa peticiones locales cortas para evitar el límite de espera del túnel. Las imágenes se muestran abajo; no se envía correo ni se modifica la base de datos de producción.


In [ ]:
import base64, io, json, time, urllib.request
from PIL import Image
from IPython.display import display
# La prueba local evita el túnel. /jobs también permite conectar Railway sin peticiones largas.
local_url = 'http://127.0.0.1:7860'
def worker_request(path, payload=None):
    request = urllib.request.Request(local_url + path, data=json.dumps(payload).encode() if payload is not None else None, headers={'Content-Type':'application/json','Authorization':'Bearer '+worker_token}, method='POST' if payload is not None else 'GET')
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)
job = worker_request('/jobs', {'name':'Cocina','description':'Menús del campamento'})
print('Generación en curso. La primera descarga puede tardar varios minutos.')
deadline = time.monotonic() + 1200
while time.monotonic() < deadline:
    progress = worker_request('/jobs/' + job['id'])
    if progress['status'] == 'ready':
        artwork = progress['result']
        break
    if progress['status'] == 'failed':
        raise RuntimeError('El modelo no pudo generar las imágenes. Revisa el registro local sin compartir claves.')
    time.sleep(3)
else:
    raise TimeoutError('La generación supera 20 minutos. El trabajo puede seguir ejecutándose; reejecuta esta celda para consultar el resultado.')
for key in ['iconUrl','mascotUrl']:
    display(Image.open(io.BytesIO(base64.b64decode(artwork[key].split(',',1)[1]))))
print('Generación comprobada. Mantén esta sesión abierta y configura la URL y clave en Railway.')


## Detener la prueba
Al terminar, detén los procesos y elimina `ARTWORK_WORKER_URL` de Railway si no vas a mantener esta sesión. Las ilustraciones ya guardadas en Neon no se pierden cuando Colab se desconecta.


In [ ]:
for process in [globals().get('tunnel'), globals().get('worker')]:
    if process is not None and process.poll() is None: process.terminate()
worker_token = ''
print('Generador y túnel detenidos.')
